# Conversión de voz cantada

Este cuaderno separa una interpretación, convierte su timbre con Seed-VC y mezcla el resultado con el acompañamiento. Usa una grabación propia o una fuente que tengas permiso de procesar.

```text
Mezcla permitida -> voz + instrumental -> conversión de timbre -> mezcla WAV
```

La melodía y el ritmo proceden de la interpretación de entrada; el modelo no crea una nueva composición.


## Preparar un entorno independiente

Seed-VC usa versiones de dependencias distintas de las del cuaderno de síntesis. Se recomienda Python 3.10 y un entorno virtual separado. Instala `requirements-canto.txt` y descarga el código de Seed-VC en una carpeta `seed-vc/` junto a este cuaderno. La celda de instalación comprueba la carpeta y la descarga si aún no existe.


In [ ]:
!pip install -r requirements-canto.txt


In [ ]:
from pathlib import Path
import subprocess

SEED_VC = Path("seed-vc")
if not (SEED_VC / "seed_vc_wrapper.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Plachtaa/seed-vc.git", str(SEED_VC)], check=True)
else:
    print("Seed-VC ya est? disponible en", SEED_VC)


In [ ]:
import logging
import subprocess
import sys
import time
import warnings
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
import torchaudio
from IPython.display import Audio, Markdown, display

from audio_pipeline import convert_voice as convertir_voz
from audio_pipeline import mix_tracks, prepare_voice_reference, separate_vocal_stems

warnings.filterwarnings("ignore")
for nombre in ("transformers", "huggingface_hub"):
    logging.getLogger(nombre).setLevel(logging.ERROR)

FS = 44_100
TRABAJO = Path("audios_canto")
TRABAJO.mkdir(parents=True, exist_ok=True)
ENTRADAS = Path("entradas")
ENTRADAS.mkdir(parents=True, exist_ok=True)
SEED_VC = Path("seed-vc")

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
PASOS = 30 if DISPOSITIVO != "cpu" else 10


def escuchar(audio, titulo=None, frecuencia=FS):
    """Reproduce audio mono o estereo."""
    if titulo:
        display(Markdown(f"**{titulo}** - {np.asarray(audio).shape[-1] / frecuencia:.1f} s"))
    display(Audio(audio, rate=frecuencia))


print("PyTorch", torch.__version__, "- dispositivo:", DISPOSITIVO, "- pasos:", PASOS)
print("Carpeta Seed-VC:", "encontrada" if (SEED_VC / "seed_vc_wrapper.py").exists() else "falta; ejecuta la celda de preparacion")

## Elegir la mezcla de entrada

Coloca en `entradas/mezcla.wav` una mezcla propia o autorizada que contenga voz e instrumentos. También puedes cambiar `CANCION` por la ruta a otro archivo. El cuaderno separa ambas partes con Demucs antes de convertir la voz; en equipos sin GPU, empieza con un fragmento corto.


In [ ]:
CANCION = Path("entradas/mezcla.wav")
INICIO = 0
DURACION = None  # None usa el archivo completo; sin GPU, empieza con un fragmento corto

if not CANCION.is_file():
    raise FileNotFoundError(f"No encuentro {CANCION}. Coloca ahi una mezcla de audio o cambia CANCION.")

cancion, _ = librosa.load(CANCION, sr=FS, mono=False, offset=INICIO, duration=DURACION)
if cancion.ndim == 1:
    cancion = np.stack([cancion, cancion])
sf.write(TRABAJO / "01_fragmento.wav", cancion.T, FS)

escuchar(cancion, "Mezcla de entrada")

## Separar voz e instrumental

Demucs estima dos pistas a partir de la mezcla original. La separación puede dejar restos de voz o instrumentos: comprueba ambas pistas antes de convertirlas.


In [ ]:
inicio = time.time()
ruta_voz, ruta_instrumental = separate_vocal_stems(
    TRABAJO / "01_fragmento.wav", TRABAJO / "separado"
)
voz_original, _ = librosa.load(ruta_voz, sr=FS, mono=True)
instrumental, _ = librosa.load(ruta_instrumental, sr=FS, mono=False)
sf.write(TRABAJO / "02_voz_original.wav", voz_original, FS)

print(f"Separacion lista en {time.time() - inicio:.0f} s")
escuchar(voz_original, "Voz separada")
escuchar(instrumental, "Acompanamiento")

Escucha la voz aislada y el acompañamiento. Si hay artefactos fuertes, prueba otro fragmento o una fuente más limpia; la conversión posterior no corrige una separación deficiente.


In [ ]:
MI_VOZ = Path("entradas/mi_voz.wav")
MI_VOZ.parent.mkdir(parents=True, exist_ok=True)
SEGUNDOS = 15
IMPORTAR = None  # ruta a una grabacion propia; None usa el archivo comun o graba

if IMPORTAR:
    audio, frecuencia = librosa.load(IMPORTAR, sr=None, mono=True)
    sf.write(MI_VOZ, audio, frecuencia)
    print(f"Importado {IMPORTAR} -> {MI_VOZ}")

elif MI_VOZ.exists():
    print(f"Se usa tu referencia: {MI_VOZ}")

else:
    import sounddevice as sd

    microfono = sd.query_devices(kind="input")
    frecuencia = int(microfono["default_samplerate"])
    print("Microfono:", microfono["name"])
    for cuenta in ["3", "2", "1"]:
        print(cuenta, end="... ", flush=True)
        time.sleep(1)
    print("Habla ahora")
    grabacion = sd.rec(int(SEGUNDOS * frecuencia), samplerate=frecuencia, channels=1, dtype="float32")
    for quedan in range(SEGUNDOS, 0, -1):
        print(quedan, end=" ", flush=True)
        time.sleep(1)
    sd.wait()
    sf.write(MI_VOZ, grabacion[:, 0], frecuencia)
    print(f"\nGuardado en {MI_VOZ}")

mi_voz, fs_mi_voz = librosa.load(MI_VOZ, sr=None, mono=True)
REFERENCIA_VOZ = TRABAJO / "referencia_limpia.wav"
prepare_voice_reference(MI_VOZ, REFERENCIA_VOZ)
print("Referencia limpiada y normalizada para Seed-VC:", REFERENCIA_VOZ)
escuchar(mi_voz, "Tu voz, referencia", fs_mi_voz)

## Preparar el modelo

El código se carga desde la carpeta local `seed-vc/`. En la primera ejecución también pueden descargarse pesos del modelo; revisa el espacio disponible y la licencia del proyecto.


In [ ]:
from audio_pipeline import load_seedvc_model

inicio = time.time()
motor = load_seedvc_model(SEED_VC, torch.device(DISPOSITIVO))

%matplotlib inline

print(f"\nSeed-VC cargado en {time.time() - inicio:.0f} s")

## Convertir el timbre

La función aplica Seed-VC a la voz separada usando `entradas/mi_voz.wav`. Puedes crear esa referencia grabándola en el cuaderno anterior, copiar ahí una grabación propia o indicar otra ruta en `IMPORTAR`. Si todavía no existe, este cuaderno permite grabarla. Los audios permanecen en el equipo.


In [ ]:
def convertir(fuente, referencia, semitonos=0, pasos=PASOS, guardar=None):
    """Convierte el timbre y conserva la posibilidad de guardar cada variante."""
    audio = convertir_voz(
        motor, Path(fuente), Path(referencia),
        pitch_shift=semitonos, diffusion_steps=pasos,
    )
    if guardar:
        sf.write(TRABAJO / guardar, audio, FS)
    return audio


inicio = time.time()
voz_convertida = convertir(
    TRABAJO / "02_voz_original.wav", REFERENCIA_VOZ, guardar="03_voz_convertida.wav"
)
print(f"{len(voz_convertida) / FS:.0f} s convertidos en {time.time() - inicio:.0f} s")

escuchar(voz_original, "La voz original")
escuchar(voz_convertida, "La misma interpretacion, con tu timbre")

Compara la salida con la voz original. El contenido musical, el fraseo y la afinación de origen se conservan en gran medida; los artefactos pueden aumentar con mezclas densas o referencias cortas.


In [ ]:
def melodia(audio, frecuencia):
    """Altura de la voz, en hercios, cada centésima de segundo. Donde no hay voz, NaN."""
    audio_16k = librosa.resample(np.asarray(audio, dtype="float32"), orig_sr=frecuencia, target_sr=16_000)
    tono = np.asarray(motor.rmvpe.infer_from_audio(audio_16k, thred=0.03), dtype="float32")
    tono[tono < 1] = np.nan
    return tono


tono_cancion = melodia(voz_original, FS)
tono_mi_voz = melodia(mi_voz, fs_mi_voz)

print(f"La voz de la canción canta alrededor de {np.nanmedian(tono_cancion):.0f} Hz")
print(f"Tú hablas alrededor de {np.nanmedian(tono_mi_voz):.0f} Hz")
print(f"Diferencia: {12 * np.log2(np.nanmedian(tono_cancion) / np.nanmedian(tono_mi_voz)):+.0f} semitonos")

## Ajustar el registro

La transposición cambia la altura de la voz sin cambiar la referencia de timbre. Prueba pequeños desplazamientos y elige mediante escucha; una octava completa puede sonar artificial.


In [ ]:
voz_octava_abajo = convertir(TRABAJO / "02_voz_original.wav", MI_VOZ, semitonos=-12, guardar="04_voz_convertida_octava_abajo.wav")

escuchar(voz_convertida, "Con tu timbre, en el tono original")
escuchar(voz_octava_abajo, "Con tu timbre, una octava más grave")

La gráfica compara las curvas de tono estimadas. Sirve para detectar cambios grandes, pero los segmentos sin voz y las estimaciones erróneas pueden afectar al dibujo.


In [ ]:
fig, eje = plt.subplots(figsize=(9, 3.4))
for audio, nombre, color in [(voz_original, "Voz original", "#8A93A8"),
                             (voz_convertida, "Convertida, tono original", "#0147FF"),
                             (voz_octava_abajo, "Convertida, una octava abajo", "#F59E0B")]:
    tono = melodia(audio, FS)
    eje.plot(np.arange(len(tono)) / 100, tono, linewidth=1.6 if nombre == "Voz original" else 1, color=color, label=nombre)
eje.set_yscale("log", base=2)
eje.set_yticks([110, 165, 220, 330, 440, 660], ["110", "165", "220", "330", "440", "660"])
eje.set_xlabel("segundos")
eje.set_ylabel("altura (Hz)")
eje.set_title("La melodía es la misma: solo cambia el timbre, y la octava si se lo pides", loc="left", fontweight="bold", fontsize=10)
eje.legend(frameon=False, fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.2))
eje.spines[["top", "right"]].set_visible(False)
plt.show()

Escoge la variante que suene más natural y asígnala a `voz_elegida`. El ejercicio final permite comparar versiones; utiliza solo voces autorizadas.


In [ ]:
voz_elegida = voz_convertida          # o voz_convertida, si te suena mejor en el tono original

## Mezclar las pistas

Se combina la voz convertida con el instrumental extraído. Ajusta `volumen_voz` para equilibrar ambas pistas y escucha el archivo completo antes de exportarlo.


In [ ]:
def mezclar(voz, instrumental, volumen_voz=1.0, guardar=None):
    """Mezcla la conversi?n con el acompa?amiento, igualando el nivel vocal original."""
    mezcla = mix_tracks(
        voz, voz_original, instrumental, voice_volume=volumen_voz
    )
    if guardar:
        sf.write(TRABAJO / guardar, mezcla.T, FS)
    return mezcla


resultado = mezclar(voz_elegida, instrumental, guardar="05_cancion_con_mi_voz.wav")

escuchar(cancion, "La cancion original")
escuchar(resultado, "La cancion, con tu voz")

El resultado se guarda en `audios_canto/05_cancion_con_mi_voz.wav`. Esa carpeta contiene datos personales o derivados y está excluida del control de versiones.


In [ ]:
def huella(audio, frecuencia):
    """Vector que resume el timbre de una voz, calculado con el codificador de hablante de Seed-VC."""
    audio_16k = torch.tensor(librosa.resample(np.asarray(audio, dtype="float32"), orig_sr=frecuencia, target_sr=16_000))
    rasgos = torchaudio.compliance.kaldi.fbank(audio_16k.unsqueeze(0).to(motor.device), num_mel_bins=80, dither=0, sample_frequency=16_000)
    rasgos = rasgos - rasgos.mean(dim=0, keepdim=True)
    with torch.no_grad():
        vector = motor.campplus_model(rasgos.unsqueeze(0))[0].float().cpu().numpy()
    return vector / np.linalg.norm(vector)


def diferencia_de_melodia(audio):
    """Mediana y dispersión, en semitonos, de la diferencia de altura con la voz original."""
    a, b = melodia(audio, FS), tono_cancion
    n = min(len(a), len(b))
    semitonos = 12 * np.log2(a[:n] / b[:n])
    return np.nanmedian(semitonos), np.nanstd(semitonos)


tu_timbre = huella(mi_voz, fs_mi_voz)

print("                                  parecido con tu voz    melodía frente a la original")
print(f"  Voz original                          {float(tu_timbre @ huella(voz_original, FS)):5.2f}                      —")
for nombre, audio in [("Convertida, tono original", voz_convertida), ("Convertida, octava abajo", voz_octava_abajo)]:
    mediana, dispersion = diferencia_de_melodia(audio)
    print(f"  {nombre:28}          {float(tu_timbre @ huella(audio, FS)):5.2f}            {mediana:+5.1f} semitonos (±{dispersion:.1f})")

## Qué medir y qué no

Las medidas de tono y timbre ayudan a comparar entradas y salidas, pero no certifican autoría, identidad ni calidad artística. Interpreta los valores junto con la escucha.


In [ ]:
MI_CANTO = Path("mi_canto.wav")
OTRA_VOZ = Path("otra_voz.wav")
GRABAR_CANTO = False
SEGUNDOS_DE_CANTO = 12

if GRABAR_CANTO:
    import sounddevice as sd

    frecuencia = int(sd.query_devices(kind="input")["default_samplerate"])
    for cuenta in ["3", "2", "1"]:
        print(cuenta, end=" … ", flush=True)
        time.sleep(1)
    print("¡CANTA!")
    grabacion = sd.rec(int(SEGUNDOS_DE_CANTO * frecuencia), samplerate=frecuencia, channels=1, dtype="float32")
    sd.wait()
    sf.write(MI_CANTO, grabacion[:, 0], frecuencia)
    print("Guardado en", MI_CANTO)

if MI_CANTO.exists() and OTRA_VOZ.exists():
    mi_canto, _ = librosa.load(MI_CANTO, sr=FS, mono=True)
    con_otra_voz = convertir(MI_CANTO, OTRA_VOZ, guardar="06_mi_canto_con_otra_voz.wav")
    escuchar(mi_canto, "Tú, cantando")
    escuchar(con_otra_voz, "Lo mismo, con la otra voz")
else:
    print("Para este paso hacen falta mi_canto.wav y otra_voz.wav junto al cuaderno.")

## Uso responsable y límites

No imites a alguien sin permiso ni publiques audio engañoso. Seed-VC y Demucs son proyectos de terceros: consulta sus licencias y condiciones antes de usar o redistribuir código, modelos o resultados.
